# Prepare Analysis Dataset

This notebook transforms the parsed Visa Bulletin dataset into an analysis-ready dataset.

The main steps are:

1. Load the 6,000-row raw dataset
2. Create `cutoff_status`
3. Convert `raw_cutoff_value` into `cutoff_date`
4. Create movement-based variables
5. Create stagnation and Current streak variables
6. Validate the final dataset
7. Save the analysis-ready dataset

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# Find the main project folder.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load the structured dataset created in the parsing notebook.
raw_path = PROCESSED_DIR / "visa_bulletins_6000_raw.csv"

df = pd.read_csv(
    raw_path,
    parse_dates=["bulletin_date"]
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (6000, 5)


,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11
1,2016-10-31,F1,China,Dates for Filing,01JAN11
2,2016-10-31,F1,India,Dates for Filing,01JAN11
3,2016-10-31,F1,Mexico,Dates for Filing,01JUN95
4,2016-10-31,F1,Philippines,Dates for Filing,01MAY06


In [3]:
# Check the data types and basic structure of the dataset.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   bulletin_date       6000 non-null   datetime64[us]
 1   visa_category       6000 non-null   str           
 2   chargeability_area  6000 non-null   str           
 3   table_type          6000 non-null   str           
 4   raw_cutoff_value    6000 non-null   str           
dtypes: datetime64[us](1), str(4)
memory usage: 234.5 KB


In [4]:
# Check the unique values in the raw cutoff column.
df["raw_cutoff_value"].value_counts().head(20)

raw_cutoff_value
C          338
01JAN17    123
01SEP17    105
22SEP15     95
22APR15     70
01DEC14     64
01OCT13     63
22NOV08     60
22OCT11     52
22NOV15     50
01SEP23     50
22JUL12     45
01MAR08     43
01OCT07     42
01DEC05     41
01MAR12     41
01DEC17     40
22SEP16     40
22MAR07     40
01SEP11     39
Name: count, dtype: int64

In [5]:
# Create a categorical status based on the original cutoff value.
# C = Current
# U = Unavailable
# All other values represent an actual cutoff date.

def get_cutoff_status(value):
    if value == "C":
        return "Current"
    elif value == "U":
        return "Unavailable"
    else:
        return "Date"


df["cutoff_status"] = df["raw_cutoff_value"].apply(get_cutoff_status)

In [11]:
# Check how many records belong to each cutoff status.
df["cutoff_status"].value_counts()


cutoff_status
Date       5662
Current     338
Name: count, dtype: int64

In [13]:
df.head()


,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_status
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date
1,2016-10-31,F1,China,Dates for Filing,01JAN11,Date
2,2016-10-31,F1,India,Dates for Filing,01JAN11,Date
3,2016-10-31,F1,Mexico,Dates for Filing,01JUN95,Date
4,2016-10-31,F1,Philippines,Dates for Filing,01MAY06,Date


In [14]:
#Convert the raw cutoff value into a usable date for analysis.
#
# Date values such as 22SEP09 are converted into real dates. (2009-09-22)
# Current (C) is represented by the end of the bulletin month.
# Unavailable (U) does not have a usable cutoff date, so it becomes NaT.

def get_cutoff_date(row):
    if row["cutoff_status"] == "Current":
        return row["bulletin_date"]

    elif row["cutoff_status"] == "Unavailable":
        return pd.NaT

    else:
        return pd.to_datetime(
            row["raw_cutoff_value"],
            format="%d%b%y"
        )


df["cutoff_date"] = df.apply(
    get_cutoff_date,
    axis=1
)

In [15]:
# Check the new cutoff_date column.
df[
    [
        "bulletin_date",
        "raw_cutoff_value",
        "cutoff_status",
        "cutoff_date"
    ]
].head(20)

,bulletin_date,raw_cutoff_value,cutoff_status,cutoff_date
0,2016-10-31,01JAN11,Date,2011-01-01
1,2016-10-31,01JAN11,Date,2011-01-01
2,2016-10-31,01JAN11,Date,2011-01-01
3,2016-10-31,01JUN95,Date,1995-06-01
4,2016-10-31,01MAY06,Date,2006-05-01
5,2016-10-31,22NOV15,Date,2015-11-22
6,2016-10-31,22NOV15,Date,2015-11-22
7,2016-10-31,22NOV15,Date,2015-11-22
8,2016-10-31,22NOV15,Date,2015-11-22
9,2016-10-31,22NOV15,Date,2015-11-22


In [18]:
df["cutoff_date"].dtype

dtype('<M8[us]')

In [19]:
# Calculate how many days the cutoff date is behind the bulletin date.
#
# Example:
# bulletin_date = 2020-06-30
# cutoff_date   = 2019-10-01
# days_behind_bulletin = 273
#
# For Current (C), cutoff_date equals bulletin_date,
# so days_behind_bulletin will be 0.
#
# For Unavailable (U), cutoff_date is NaT,
# so the result will remain missing.

df["days_behind_bulletin"] = (
    df["bulletin_date"] - df["cutoff_date"]
).dt.days

In [25]:
# Preview the new variable.
df[
    [
        "bulletin_date",
        "visa_category",
        "chargeability_area",
        "table_type",
        "raw_cutoff_value",
        "cutoff_status",
        "cutoff_date",
        "days_behind_bulletin"
    ]
].head(20)

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_status,cutoff_date,days_behind_bulletin
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2130
1,2016-10-31,F1,China,Dates for Filing,01JAN11,Date,2011-01-01,2130
2,2016-10-31,F1,India,Dates for Filing,01JAN11,Date,2011-01-01,2130
3,2016-10-31,F1,Mexico,Dates for Filing,01JUN95,Date,1995-06-01,7823
4,2016-10-31,F1,Philippines,Dates for Filing,01MAY06,Date,2006-05-01,3836
5,2016-10-31,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,22NOV15,Date,2015-11-22,344
6,2016-10-31,F2A,China,Dates for Filing,22NOV15,Date,2015-11-22,344
7,2016-10-31,F2A,India,Dates for Filing,22NOV15,Date,2015-11-22,344
8,2016-10-31,F2A,Mexico,Dates for Filing,22NOV15,Date,2015-11-22,344
9,2016-10-31,F2A,Philippines,Dates for Filing,22NOV15,Date,2015-11-22,344


In [26]:
# Current records should have 0 days behind the bulletin.
df[
    df["cutoff_status"] == "Current"
][
    [
        "bulletin_date",
        "raw_cutoff_value",
        "cutoff_date",
        "days_behind_bulletin"
    ]
].head()

,bulletin_date,raw_cutoff_value,cutoff_date,days_behind_bulletin
1680,2019-07-31,C,2019-07-31,0
1681,2019-07-31,C,2019-07-31,0
1682,2019-07-31,C,2019-07-31,0
1683,2019-07-31,C,2019-07-31,0
1684,2019-07-31,C,2019-07-31,0


In [27]:
# Sort the dataset so monthly movement is calculated in chronological order
# within the same visa category, chargeability area, and table type.
df = df.sort_values(
    by=[
        "visa_category",
        "chargeability_area",
        "table_type",
        "bulletin_date"
    ]
).reset_index(drop=True)

In [28]:
# Calculate the month-to-month change in cutoff_date.
# Positive = moved forward
# Zero = no movement
# Negative = retrogressed
#
# The first month in each group has no previous month,
# so movement_days will be missing for that row.

df["movement_days"] = (
    df.groupby(
        ["visa_category", "chargeability_area", "table_type"]
    )["cutoff_date"]
    .diff()
    .dt.days
)

In [31]:
# Preview the new movement variable.
df[
    [
        "bulletin_date",
        "visa_category",
        "chargeability_area",
        "table_type",
        "raw_cutoff_value",
        "cutoff_date",
        "movement_days"
    ]
].head(300)

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_date,movement_days
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,NaN
1,2016-11-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0
2,2016-12-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0
3,2017-01-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0
4,2017-02-28,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0
...,...,...,...,...,...,...,...
295,2021-05-31,F1,China,Dates for Filing,01OCT15,2015-10-01,16.0
296,2021-06-30,F1,China,Dates for Filing,01MAR16,2016-03-01,152.0
297,2021-07-31,F1,China,Dates for Filing,15MAY16,2016-05-15,75.0
298,2021-08-31,F1,China,Dates for Filing,15MAY16,2016-05-15,0.0


In [32]:
# Convert movement_days into a categorical movement status.
#
# Positive movement = Advance
# Zero movement = No Movement
# Negative movement = Retrogress
# Missing movement_days stays missing.

def get_movement_status(value):
    if pd.isna(value):
        return pd.NA
    elif value > 0:
        return "Advance"
    elif value == 0:
        return "No Movement"
    else:
        return "Retrogress"


df["movement_status"] = df["movement_days"].apply(get_movement_status)

In [33]:
# Check how many records fall into each movement category.
df["movement_status"].value_counts(dropna=False)

movement_status
No Movement    3068
Advance        2852
NaN              50
Retrogress       30
Name: count, dtype: int64

In [34]:
df[
    [
        "bulletin_date",
        "visa_category",
        "chargeability_area",
        "table_type",
        "raw_cutoff_value",
        "cutoff_date",
        "movement_days",
        "movement_status"
    ]
].head(300)

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_date,movement_days,movement_status
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,NaN,NaN
1,2016-11-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0,No Movement
2,2016-12-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0,No Movement
3,2017-01-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0,No Movement
4,2017-02-28,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,2011-01-01,0.0,No Movement
...,...,...,...,...,...,...,...,...
295,2021-05-31,F1,China,Dates for Filing,01OCT15,2015-10-01,16.0,Advance
296,2021-06-30,F1,China,Dates for Filing,01MAR16,2016-03-01,152.0,Advance
297,2021-07-31,F1,China,Dates for Filing,15MAY16,2016-05-15,75.0,Advance
298,2021-08-31,F1,China,Dates for Filing,15MAY16,2016-05-15,0.0,No Movement


In [35]:
# Count how many consecutive months the cutoff date has remained unchanged.
#
# The first month with no movement is counted as 1.
# The count resets to 0 when the cutoff date changes.
# Current periods are tracked separately with current_streak.

def count_stagnant_months(series):
    count = 0
    result = []

    for value in series:
        if value == 0:
            count += 1
        else:
            count = 0

        result.append(count)

    return result


df["months_stagnant"] = (
    df.groupby(
        ["visa_category", "chargeability_area", "table_type"]
    )["movement_days"]
    .transform(count_stagnant_months)
)

In [36]:
# Preview records that have remained unchanged for at least one month.
df[
    df["months_stagnant"] > 0
][
    [
        "bulletin_date",
        "visa_category",
        "chargeability_area",
        "table_type",
        "raw_cutoff_value",
        "movement_days",
        "months_stagnant"
    ]
].head(20)

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,movement_days,months_stagnant
1,2016-11-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,0.0,1
2,2016-12-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,0.0,2
3,2017-01-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,0.0,3
4,2017-02-28,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,0.0,4
5,2017-03-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,0.0,5
6,2017-04-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,0.0,6
8,2017-06-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,22JUL11,0.0,1
9,2017-07-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,22JUL11,0.0,2
10,2017-08-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,22JUL11,0.0,3
11,2017-09-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,22JUL11,0.0,4


In [37]:
# Count how many consecutive months the category has remained Current.
#
# The first Current month is counted as 1.
# The count increases while Current continues.
# The count resets to 0 when the status is no longer Current.

def count_current_streak(series):
    count = 0
    result = []

    for status in series:
        if status == "Current":
            count += 1
        else:
            count = 0

        result.append(count)

    return result


df["current_streak"] = (
    df.groupby(
        ["visa_category", "chargeability_area", "table_type"]
    )["cutoff_status"]
    .transform(count_current_streak)
)

In [40]:
# Preview records that are part of a Current streak.
df[
    df["current_streak"] > 0
][
    [
        "bulletin_date",
        "visa_category",
        "chargeability_area",
        "table_type",
        "raw_cutoff_value",
        "cutoff_date",
        "cutoff_status",
        "movement_days",
        "current_streak"
    ]
].head(20)

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_date,cutoff_status,movement_days,current_streak
1266,2022-04-30,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-04-30,Current,150.0,1
1267,2022-05-31,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-05-31,Current,31.0,2
1268,2022-06-30,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-06-30,Current,30.0,3
1269,2022-07-31,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-07-31,Current,31.0,4
1270,2022-08-31,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-08-31,Current,31.0,5
1271,2022-09-30,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-09-30,Current,30.0,6
1272,2022-10-31,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-10-31,Current,31.0,7
1273,2022-11-30,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-11-30,Current,30.0,8
1274,2022-12-31,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2022-12-31,Current,31.0,9
1275,2023-01-31,F2A,All Chargeability Areas Except Those Listed,Dates for Filing,C,2023-01-31,Current,31.0,10


In [41]:
# Check the final dataset before saving.

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

Dataset shape: (6000, 12)

Columns:
['bulletin_date', 'visa_category', 'chargeability_area', 'table_type', 'raw_cutoff_value', 'cutoff_status', 'cutoff_date', 'days_behind_bulletin', 'movement_days', 'movement_status', 'months_stagnant', 'current_streak']

Missing values:
bulletin_date            0
visa_category            0
chargeability_area       0
table_type               0
raw_cutoff_value         0
cutoff_status            0
cutoff_date              0
days_behind_bulletin     0
movement_days           50
movement_status         50
months_stagnant          0
current_streak           0
dtype: int64


In [42]:
# Perform final logical checks before saving the dataset.

print("Cutoff status counts:")
print(df["cutoff_status"].value_counts())

print("\nMovement status counts:")
print(df["movement_status"].value_counts(dropna=False))

print("\nNegative days behind bulletin:")
print((df["days_behind_bulletin"] < 0).sum())

print("\nDuplicate records:")
print(
    df.duplicated(
        subset=[
            "bulletin_date",
            "visa_category",
            "chargeability_area",
            "table_type"
        ]
    ).sum()
)

print("\nMaximum stagnant streak:")
print(df["months_stagnant"].max())

print("\nMaximum Current streak:")
print(df["current_streak"].max())

Cutoff status counts:
cutoff_status
Date       5662
Current     338
Name: count, dtype: int64

Movement status counts:
movement_status
No Movement    3068
Advance        2852
NaN              50
Retrogress       30
Name: count, dtype: int64

Negative days behind bulletin:
0

Duplicate records:
0

Maximum stagnant streak:
62

Maximum Current streak:
45


In [43]:
# Save the complete transformed dataset.
# This version keeps all calculated and derived variables.

full_output_path = (
    PROCESSED_DIR / "visa_bulletins_transformed_full.csv"
)

df.to_csv(
    full_output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved to:", full_output_path)
print("Dataset shape:", df.shape)

Saved to: /Users/serratak/Desktop/visa-bulletin-project/visa-bulletin-project/data/processed/visa_bulletins_transformed_full.csv
Dataset shape: (6000, 12)


In [44]:
analysis_df = df[
    [
        "bulletin_date",
        "visa_category",
        "chargeability_area",
        "table_type",
        "cutoff_date",
        "cutoff_status",
        "movement_days"
    ]
].copy()

In [45]:
analysis_output_path = (
    PROCESSED_DIR / "visa_bulletins_analysisQ1.csv"
)

analysis_df.to_csv(
    analysis_output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved to:", analysis_output_path)
print("Dataset shape:", analysis_df.shape)

Saved to: /Users/serratak/Desktop/visa-bulletin-project/visa-bulletin-project/data/processed/visa_bulletins_analysisQ1.csv
Dataset shape: (6000, 7)
